# 10 — Knowledge map: BKT on ASSISTments, attachment and links on the demo account

The knowledge map shows, per concept, how well a student knows it. Two signals feed it: **how the student answered**
(this notebook) and **how the memory forecast says the concept is decaying** (notebook 08). Bayesian Knowledge Tracing
(BKT; Corbett & Anderson 1995) is the model for the first: a concept is either known or not, each answer is right with
probability 1 - slip if it is known and with probability guess if not, and after every answer the student may learn it.
The state is the probability that the concept is known. Choice questions are easier to guess than written ones, so the
two kinds have their own guess parameter (`ml_pipeline/bkt.py`).

**Data.** ASSISTments 2009-2010 skill builder (Feng, Heffernan & Koedinger 2009; mirror `Atomi/ASSISTments2009`; see
`datasets/README.md`): 4,148 students, 274,331 responses, 139 skills, each response with its answer type. Students are
split 80/20 by student with `config.RANDOM_SEED`, so the test students are never seen when fitting. The sequence of
one student on one skill is one BKT sequence.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from ml_pipeline import config, bkt

raw = pd.read_parquet(config.REPO_ROOT / 'datasets' / 'raw' / 'ASSISTments' / 'assist2009.parquet')
print(f"{len(raw)} students, {sum(len(x) for x in raw.grades)} responses, "
      f"{len({s for x in raw.skill_ids for s in x})} skills")

def sequences_of(row):
    by = {}
    for skill, grade, atype in zip(row.skill_ids, row.grades, row.answer_types):
        if atype == 'open_response':      # 8 answers, graded by hand: not a right/wrong event
            continue
        by.setdefault(skill, []).append((grade == '1', bkt.kind_of(atype)))
    return list(by.values())

rng = np.random.default_rng(config.RANDOM_SEED)
order = rng.permutation(len(raw))
cut = int(0.8 * len(raw))
train_students, test_students = raw.iloc[order[:cut]], raw.iloc[order[cut:]]
train = [s for r in train_students.itertuples() for s in sequences_of(r)]
test = [s for r in test_students.itertuples() for s in sequences_of(r)]
for name, seqs, students in [('train', train, train_students), ('test', test, test_students)]:
    print(f"{name}: {len(students)} students, {len(seqs)} (student, skill) sequences, "
          f"{sum(len(s) for s in seqs)} answers")
kinds = pd.Series([k for s in train for _, k in s]).value_counts()
print('train answers by kind:', kinds.to_dict())

4148 students, 274331 responses, 139 skills
train: 3318 students, 33502 (student, skill) sequences, 227069 answers
test: 830 students, 7274 (student, skill) sequences, 47254 answers
train answers by kind: {'written': 202088, 'choice': 24981}


## Fit

`bkt.fit` maximises the likelihood of the training sequences with L-BFGS-B, with each parameter bounded to a sensible
range. The likelihood is plain Python, so the first cell below times one evaluation: if a fit would take more than
about three minutes the training students are subsampled (1,500), otherwise all of them are used.

In [2]:
t0 = time.time()
bkt.neg_log_likelihood(train, bkt.Params(0.3, 0.15, 0.1, 0.25, 0.1))
one_eval = time.time() - t0
# L-BFGS-B with finite differences costs 6 evaluations per iteration; 100 iterations is a generous ceiling.
estimate = one_eval * 6 * 100
print(f"one likelihood evaluation: {one_eval:.2f}s; worst-case fit about {estimate / 60:.1f} min")

fit_students = train_students
if estimate > 180:
    fit_students = train_students.sample(1500, random_state=config.RANDOM_SEED)
    print('subsampled to 1,500 training students')
fit_seqs = [s for r in fit_students.itertuples() for s in sequences_of(r)]

t0 = time.time()
params = bkt.fit(fit_seqs)
elapsed = time.time() - t0
print(f"fitted on {len(fit_students)} students, {len(fit_seqs)} sequences, "
      f"{sum(len(s) for s in fit_seqs)} answers in {elapsed:.0f}s")
pd.Series(params.__dict__, name='fitted').to_frame()

one likelihood evaluation: 0.11s; worst-case fit about 1.1 min


fitted on 3318 students, 33502 sequences, 227069 answers in 22s


,fitted
init,0.740669
learn,0.055021
slip,0.204407
guess_choice,0.500000
guess_written,0.261751


## Does it predict the next answer?

For every answer of the test students, the model predicts it from the state **before** that answer, then updates.
BKT predicts with `p_correct(p_known, kind)`. The baseline is what a plain tally would say: the student's running
accuracy on that skill with a 0.6 prior, `(right so far + 0.6) / (answered so far + 1)`. AUC is computed over all test
answers; the baseline ignores the answer kind and the learning curve.

In [3]:
rows = []   # one per test answer: bkt prediction, baseline prediction, outcome, p_known before
for seq in test:
    p, right = params.init, 0
    for i, (correct, kind) in enumerate(seq):
        rows.append((bkt.p_correct(p, kind, params), (right + 0.6) / (i + 1), int(correct), p, kind))
        p = bkt.update(p, correct, kind, params)
        right += int(correct)
pred = pd.DataFrame(rows, columns=['bkt', 'baseline', 'correct', 'p_known', 'kind'])
auc = roc_auc_score(pred.correct, pred.bkt)
baseline_auc = roc_auc_score(pred.correct, pred.baseline)
print(f"{len(pred)} test answers, {pred.correct.mean():.3f} right")
print(f"AUC  BKT {auc:.4f}   running-accuracy baseline {baseline_auc:.4f}")
for kind, g in pred.groupby('kind'):
    print(f"  {kind:8s} n={len(g):6d}  BKT {roc_auc_score(g.correct, g.bkt):.4f}  baseline {roc_auc_score(g.correct, g.baseline):.4f}")

47254 test answers, 0.664 right
AUC  BKT 0.6899   running-accuracy baseline 0.6950
  choice   n=  4980  BKT 0.6895  baseline 0.6995
  written  n= 42274  BKT 0.6887  baseline 0.6938


## Thresholds: strong, shaky, weak

The map needs three bands. A concept is *strong* when `p_known` is at least the upper cut, *weak* below the lower cut,
and *shaky* between. For each candidate pair the test answers are bucketed by their prediction-time `p_known` and the
accuracy of the next answer is shown per bucket.

**Rule, fixed before looking:** keep 0.8 / 0.5 unless another pair separates the buckets clearly better. "Separates"
means the smaller of the two gaps (strong minus shaky accuracy, shaky minus weak accuracy), and a pair only counts
when every bucket holds at least 5% of the answers. Another pair replaces 0.8 / 0.5 only if its smaller gap is larger
by at least 0.03 (three accuracy points).

In [4]:
candidates = [(0.7, 0.4), (0.8, 0.5), (0.9, 0.6)]
table = []
for hi, lo in candidates:
    band = np.where(pred.p_known >= hi, 'strong', np.where(pred.p_known >= lo, 'shaky', 'weak'))
    g = pred.groupby(band).correct.agg(['mean', 'size'])
    acc = {b: g['mean'].get(b, np.nan) for b in ('strong', 'shaky', 'weak')}
    share = {b: g['size'].get(b, 0) / len(pred) for b in ('strong', 'shaky', 'weak')}
    table.append({'strong >=': hi, 'shaky >=': lo,
                  'acc strong': acc['strong'], 'acc shaky': acc['shaky'], 'acc weak': acc['weak'],
                  'share strong': share['strong'], 'share shaky': share['shaky'], 'share weak': share['weak'],
                  'min gap': min(acc['strong'] - acc['shaky'], acc['shaky'] - acc['weak']),
                  'min share': min(share.values())})
table = pd.DataFrame(table).round(3)
table

,strong >=,shaky >=,acc strong,acc shaky,acc weak,share strong,share shaky,share weak,min gap,min share
0,0.7,0.4,0.741,0.583,0.363,0.728,0.118,0.155,0.158,0.118
1,0.8,0.5,0.788,0.634,0.423,0.505,0.269,0.226,0.154,0.226
2,0.9,0.6,0.796,0.648,0.446,0.457,0.287,0.256,0.148,0.256


In [5]:
ok = table[table['min share'] >= 0.05]
default = table[(table['strong >='] == 0.8) & (table['shaky >='] == 0.5)].iloc[0]
best = ok.loc[ok['min gap'].idxmax()]
if best['min gap'] - default['min gap'] >= 0.03:
    chosen = best
else:
    chosen = default
thresholds = {'strong': float(chosen['strong >=']), 'shaky': float(chosen['shaky >='])}
print(f"default 0.8/0.5 min gap {default['min gap']:.3f}; best eligible {best['strong >=']}/{best['shaky >=']} "
      f"min gap {best['min gap']:.3f}")
print('chosen:', thresholds)

default 0.8/0.5 min gap 0.154; best eligible 0.7/0.4 min gap 0.158
chosen: {'strong': 0.8, 'shaky': 0.5}


In [6]:
bkt.save(params, thresholds, auc=round(float(auc), 4), baseline_auc=round(float(baseline_auc), 4),
         source='ASSISTments 2009, notebooks/10_knowledge_map.ipynb')
print(bkt.PATH.read_text())
assert bkt.load() == (params, thresholds)

{
 "params": {
  "init": 0.7406692765246253,
  "learn": 0.05502086979119816,
  "slip": 0.20440710734048892,
  "guess_choice": 0.5,
  "guess_written": 0.26175108970449185
 },
 "thresholds": {
  "strong": 0.8,
  "shaky": 0.5
 },
 "auc": 0.6899,
 "baseline_auc": 0.695,
 "source": "ASSISTments 2009, notebooks/10_knowledge_map.ipynb"
}



## Results and limits

- The figures above are on students the model never saw. **BKT with one shared parameter set does not beat the
  tally**: AUC 0.690 against 0.695 for the running accuracy of the same student on the same skill, and the tally is
  ahead on choice and written answers alike. Different starting values reach the same fit, so this is not a failed
  optimisation. The tally knows how strong *this* student is; the shared BKT parameters only know the average
  student, and 0.005 AUC is what that costs here. The two are close enough that BKT is kept for what the tally cannot
  give: a state that moves with each answer, a learning step, and a different weight for choice and written answers.
  It is not kept because it predicts better.
- `guess_choice` sits on its upper bound (0.5): the data would take a higher value, so the choice guess is a ceiling
  here, not an estimate. The fitted `init` (0.74) is high because most students already know the skill when they
  reach it in a skill builder.
- The data are **middle-school maths skills** answered in a tutoring system, not university subjects. The parameters
  are a prior that is reasonable for "a student answering questions on a concept"; they are not a claim about any
  particular course. Per-concept parameters are not fitted: StudyFlow concepts have too few answers each.
- The three bands are bucket labels for the map. The rule above picks the cut-offs from how well the buckets
  separate next-answer accuracy, nothing more.
- That the knowledge map shows the **weaker** of its two signals (BKT understanding, memory-forecast retention) is a
  design choice made for the product, not something fitted here.

# Part 2 — Attaching cards and questions to concepts

The map's concepts are the documents' key points (same idea in two documents merged, `MERGE_THRESHOLD`). Every card,
question and practice problem is attached to the closest concept of its own documents by TF-IDF over words and
characters, or to none below `ATTACH_THRESHOLD` (`ml_pipeline/concept_map.py`). This part checks both thresholds on
the demo account.

**Data.** `manage.py dump_concept_items --email studyflow.demo@gmail.com` writes the account's key points, merged
concepts, items and stored links to `ml_pipeline/outputs/concept_items.json` (not in git). Items of the three demo
notebooks whose document has key points were labelled by hand in `datasets/handmade/concept_attachment.json`: each
was read against the key points of its own document and given the one it is about, or `null` when none fits (the
demo documents are mixed science texts, so many items are about something no key point covers). Items whose document
has no key points cannot be attached at all and are left out of the labels.

In [7]:
import json
from collections import Counter
from ml_pipeline import concept_map
from ml_pipeline.concept_map import attach, merge_concepts, _text
from ml_pipeline.subject_match import _vectorise

dump = json.loads((config.REPO_ROOT / 'ml_pipeline' / 'outputs' / 'concept_items.json').read_text())
labels = json.loads((config.REPO_ROOT / 'datasets' / 'handmade' / 'concept_attachment.json').read_text())['items']
books = {n['id']: n for n in dump['notebooks']}
pd.DataFrame([{'notebook': f"{n['id']} {n['title']}", 'documents': len({p['source_id'] for p in n['points']}),
               'key points': len(n['points']), 'concepts': len(n['concepts']), 'items': len(n['items']),
               'labelled': sum(r['notebook'] == n['id'] for r in labels),
               'labelled null': sum(r['notebook'] == n['id'] and r['concept'] is None for r in labels)}
              for n in books.values()])

,notebook,documents,key points,concepts,items,labelled,labelled null
0,32 Biology,1,15,15,49,22,7
1,33 Chemistry,2,29,29,57,34,7
2,34 Physics,1,15,15,47,25,7


Attachment runs on all items of a notebook at once, as the app does (the TF-IDF weights depend on the whole set),
and is scored on the labelled ones. **Accuracy** counts an item right when it gets the labelled concept, `null`
included; **coverage** is the share attached to some concept; the last two columns split accuracy by label.

In [8]:
def attach_scores(t):
    got = {}
    for n in books.values():
        got.update({(n['id'], k): v for k, v in attach(n['items'], n['concepts'], threshold=t).items()})
    pred = [got[(r['notebook'], r['item'])] for r in labels]
    gold = [r['concept'] for r in labels]
    hit = [p == g for p, g in zip(pred, gold)]
    on = [h for h, g in zip(hit, gold) if g is not None]
    off = [h for h, g in zip(hit, gold) if g is None]
    return {'threshold': t, 'accuracy': np.mean(hit), 'coverage': np.mean([p is not None for p in pred]),
            'right on labelled concept': np.mean(on), 'right on null': np.mean(off)}

grid = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]
sweep = pd.DataFrame([attach_scores(t) for t in grid]).round(3)
sweep

,threshold,accuracy,coverage,right on labelled concept,right on null
0,0.05,0.704,0.963,0.900,0.143
1,0.08,0.765,0.877,0.883,0.429
2,0.10,0.827,0.815,0.883,0.667
3,0.12,0.852,0.765,0.867,0.810
4,0.15,0.815,0.728,0.817,0.810
5,0.18,0.778,0.691,0.767,0.810
6,0.20,0.728,0.630,0.700,0.810
7,0.25,0.704,0.543,0.633,0.905
8,0.30,0.679,0.481,0.583,0.952


In [9]:
best_attach = float(sweep.loc[sweep['accuracy'].idxmax(), 'threshold'])
current = sweep[sweep['threshold'] == concept_map.ATTACH_THRESHOLD].iloc[0]
print(f"best ATTACH_THRESHOLD {best_attach} (accuracy {sweep['accuracy'].max():.3f}); "
      f"current {concept_map.ATTACH_THRESHOLD} (accuracy {current['accuracy']:.3f})")
got = {}
for n in books.values():
    got.update({(n['id'], k): v for k, v in attach(n['items'], n['concepts'], threshold=best_attach).items()})
label_of = {(n['id'], c['key']): c['label'] for n in books.values() for c in n['concepts']}
wrong = [{'item': r['item'], 'text': r['text'][:70], 'labelled': label_of.get((r['notebook'], r['concept'])),
          'attached': label_of.get((r['notebook'], got[(r['notebook'], r['item'])]))}
         for r in labels if got[(r['notebook'], r['item'])] != r['concept']]
pd.DataFrame(wrong)

best ATTACH_THRESHOLD 0.12 (accuracy 0.852); current 0.12 (accuracy 0.852)


,item,text,labelled,attached
0,card:704,___ is our species. Homo sapiens,NaN,Natural Selection
1,card:701,The fossil record shows that this type of even...,Evidence for Evolution,Overpopulation and Extinction
2,card:699,How do some animals change their depth? By cha...,NaN,Nutrient Cycles
3,card:696,Photosynthetic protists use ___ to make food. ...,NaN,Evolution Changes Traits
4,q:513,Ferns are the most common seedless vascular pl...,Plants Sustain Life,Climate Change Impacts
5,card:765,Which substance produces positive hydrogen ion...,Brønsted-Lowry Acids and Bases,Acid Strength
6,card:755,In what two types of geological areas is boron...,Boron Reactivity,NaN
7,card:713,What does lattice energy of an ionic solid mea...,Ionic Bonding,NaN
8,q:541,Many adults and some children suffer from a de...,Enzyme Catalysis,System Enthalpy
9,card:744,Figure 8.11 The sun emits energy in the form o...,Sun's Energy Source,How Generators Work


### Merging key points

Two key points merge when they come from different documents and their similarity reaches `MERGE_THRESHOLD`. The
table counts merges at each value over the demo key points, and lists the closest pairs from different documents for
a reader to judge. Only Chemistry has two documents with key points, so only its pairs can merge. The closest pairs
inside one document are shown too: they are distinct ideas of one text, so they show how similar two concepts can be
while still being different.

In [10]:
def pair_sims(points, same_doc):
    m = _vectorise([_text(p) for p in points])
    s = (m @ m.T).toarray()
    return sorted(((float(s[i, j]), points[j]['label'], points[i]['label'])
                   for i in range(len(points)) for j in range(i)
                   if (points[i]['source_id'] == points[j]['source_id']) == same_doc), reverse=True)

merge_rows = []
for t in [0.15, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:
    concept_map.MERGE_THRESHOLD = t
    merged = sum(len(n['points']) - len(merge_concepts(n['points'])) for n in books.values())
    merge_rows.append({'threshold': t, 'key points merged away': merged})
concept_map.MERGE_THRESHOLD = 0.45
display(pd.DataFrame(merge_rows))
cross = [(round(s, 3), a, b) for n in books.values() for s, a, b in pair_sims(n['points'], False)][:8] if any(
    len({p['source_id'] for p in n['points']}) > 1 for n in books.values()) else []
same = sorted((round(s, 3), a, b) for n in books.values() if n['points'] for s, a, b in pair_sims(n['points'], True))[::-1][:5]
display(pd.DataFrame(cross, columns=['similarity', 'key point', 'other document']))
display(pd.DataFrame(same, columns=['similarity', 'key point', 'same document']))

,threshold,key points merged away
0,0.15,1
1,0.20,0
2,0.30,0
3,0.35,0
4,0.40,0
5,0.45,0
6,0.50,0
7,0.55,0


,similarity,key point,other document
0,0.161,Atomic Mass and Isotopes,Nuclear Chain Reactions
1,0.150,Compounds and Chemical Bonds,Protein Composition
2,0.149,Polarity and Dissolution,Acid Strength
3,0.127,ATP Energy Release,Photosynthesis and Respiration
4,0.125,Diverse Carbides,Boron Reactivity
5,0.124,ATP Energy Release,Cellular Energy Needs
6,0.122,Ions as Electrolytes,Acid Strength
7,0.120,Atomic Mass and Isotopes,Brønsted-Lowry Acids and Bases


,similarity,key point,same document
0,0.388,Natural Selection,Population Limits
1,0.241,Evolution Changes Traits,Natural Selection
2,0.202,Ionic Bonding,Polarity and Dissolution
3,0.196,Ionic Bonding,Electron Energy Levels and Orbitals
4,0.188,EM Wave Energy and Frequency,Voltage and Energy Difference


# Part 3 — Links between concepts

Links come from one model request per notebook (`find_links`), cleaned of loops, unknown numbers and more than three
prerequisites per concept. There is no gold standard, so they are listed for a reader to judge.

In [11]:
for n in books.values():
    edges = (n['links'] or {}).get('edges', [])
    lab = {c['key']: c['label'] for c in n['concepts']}
    into = Counter(e['to'] for e in edges)
    print(f"#{n['id']} {n['title']}: {len(n['concepts'])} concepts, {len(edges)} links, "
          f"{sum(1 for c in n['concepts'] if c['key'] in into or any(e['from'] == c['key'] for e in edges))} concepts linked")
    for e in edges:
        print(f"   {lab.get(e['from'], e['from'])}  ->  {lab.get(e['to'], e['to'])}   ({e['why']})")

#32 Biology: 15 concepts, 10 links, 12 concepts linked
   Evolution Changes Traits  ->  Natural Selection   (Natural selection is a primary mechanism of evolution.)
   Evolution Changes Traits  ->  Speciation Processes   (Speciation describes how new species arise, a fundamental outcome of evolution.)
   Evolution Changes Traits  ->  Evidence for Evolution   (Evidence for evolution demonstrates that the process of evolution occurs.)
   Natural Selection  ->  Competition and Adaptation   (Natural selection is the process by which competition leads to adaptation and trait changes.)
   Plants Sustain Life  ->  Nutrient Cycles   (Plants are primary producers and organisms through which nutrients cycle in ecosystems.)
   Nutrient Cycles  ->  Wetlands Filter Runoff   (Wetlands purify water by processing and cycling nutrients, a specific application of nutrient cycling.)
   Population Limits  ->  Climate Change Impacts   (Climate change alters environmental conditions, directly impacting the 

## Results and limits (parts 2 and 3)

- **Attachment.** On 81 labelled items (21 labelled `null`) the current `ATTACH_THRESHOLD` of 0.12 is
  also the best value of the grid: 85% of items get the labelled concept, with 77% attached to some concept. Lower
  values attach nearly everything and so get most `null` items wrong (0.05: 70%); higher values start dropping
  items that do belong (0.20: 73%). The constant is kept. Of the 12 mistakes left, 6 go to the wrong key point, often one
  that shares a word or two (a card on the Sun's radiation goes to generators, a question on ferns to climate
  change); 4 are items about nothing the document's key points cover that are attached anyway; 2 belong to a key
  point but stay below the threshold (boron, lattice energy). Word overlap cannot do much better on short cards.
- **Merging.** The demo account cannot test `MERGE_THRESHOLD` well. Its only notebook with two read
  documents (Chemistry: "Atoms and bonding" and "Chemical reactions") has no idea in both, and the closest pair across
  the two documents is at 0.16, so nothing merges at any value above that, 0.45 included. Inside one document,
  distinct ideas reach 0.39 (Biology), so a threshold much below 0.45 would start merging different concepts. The
  constant (lowered from 0.55 when a covalent-bond test pair in two wordings scored 0.48) is kept; it needs two
  documents that cover the same idea to be checked properly.
- **Links.** Physics got 6 links and Biology 10; the Chemistry request failed twice (the model was
  busy, then the daily free quota ran out), so Chemistry has none yet. Most links read as sensible (evolution before
  natural selection and speciation, convection before air as an insulator). Two are doubtful: "How Generators Work
  before Benefits of Solar Energy" (solar panels do not work by induction) and "Population Limits before Climate
  Change Impacts" (the dependency runs the other way, if at all). The prompt asks for clear, direct dependencies
  only; the map shows each link's reason so a student can judge it.
- **Limits.** One account, three notebooks of mixed science texts, and labels by one person (the author), read
  against the same key points the method sees. The labels have not been checked by a second reader, and the
  thresholds are picked on the same items they are scored on, so the accuracy is optimistic. Keys of the form
  `<source>:<index>` follow the stored key points: if a document is read again, the labels must be redone. Items of
  documents without key points (two of the six demo documents, where the model request timed out) are not
  covered.